# Yelp Review Star Rating Prediction using BERT + MLP

This notebook trains a neural network to predict **Yelp star ratings (1–5)** from cleaned customer review text.  
It combines **BERT** (for text embeddings) with a **Multi-Layer Perceptron (MLP)** classifier to perform multi-class classification.

### Overview
- Load and clean Yelp review data using `load_data()` function  
- Tokenize text with **BERT tokenizer** (`bert-base-uncased`)  
- Extract BERT embeddings and feed them into a **2-layer MLP**  
- Train and evaluate the model on predicting **1–5 star ratings**  

### Output
- Trained BERT+MLP model for 5-class classification  
- Classification report and accuracy score on test data  

---


In [22]:
import os
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch # may need to run ! pip install torch in a separate cell (see below)
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

In [23]:
! pip install torch


In [24]:
def load_data():

    folder = "../Preprocessing-FeatureExtraction/cleaned-data/"
    csv_files = glob.glob(os.path.join(folder, "*.csv"))
    
    dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df['state'] = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df)
    
    data = pd.concat(dfs, ignore_index=True)
    df = data.dropna()
    
    # Create sentiment labels: 0=negative (1-2), 1=neutral (3), 2=positive (4-5)
    df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

    # Labeling star ratings as 0-4 instead of 1-5 (for CrossEntropyLoss)
    df['label'] = df['stars'] - 1

    print(f"Loaded {len(df)} reviews from {len(dfs)} states.")
    return df

In [25]:
# Split data
df = load_data()
sample = df.sample(500000)
train_texts, test_texts, y_train, y_test = train_test_split(
    sample['clean_text'].tolist(),
    sample['label'].tolist(),
    test_size=0.1,
    random_state=42,
    stratify=sample['label']
)

/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_5691/2003529814.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['sentiment'] = df['stars'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))
/var/folders/cl/5mfvhcls2nv2h9gy15m04b640000gn/T/ipykernel_5691/2003529814.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['stars'] - 1


Loaded 5222860 reviews from 20 states.


In [26]:
# Tokenize (BERT)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 128

def tokenize_batch(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt'
    )

/Users/juliasober/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [27]:
# Tokenize dataset
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_batch(texts)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = ReviewDataset(train_texts, y_train)
test_dataset = ReviewDataset(test_texts, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [31]:
# Define model
class BERT_MLP(nn.Module):
    def __init__(self, num_classes=5):
        super(BERT_MLP, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():  # freeze BERT weights if you only want to train MLP
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(cls_output)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BERT_MLP(num_classes=5).to(device)

/Users/juliasober/anaconda3/lib/python3.11/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [32]:
# Train model
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Training loss: {avg_loss:.4f}")

Training Epoch 1:  11%|█         | 2967/28125 [1:37:06<13:43:24,  1.96s/it]


KeyboardInterrupt: 

In [ ]:
# Evaluate
model.eval()
preds, true_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, dim=1)
        preds.extend(predicted.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

print("\nClassification Report:")
print(classification_report(true_labels, preds))
print("Accuracy:", accuracy_score(true_labels, preds))